# MLColvar integration for TICA

Here we prepare heavily-weighted data phasepoints generated from TIS paths to be optimally ingested by `mlcolvar`.

## 1. Import the necessary functions

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib qt

import sys
import os

# Clean sys.path from 'inftools' which has a shadowing 'tistools' folder 
# turning the tistools package into an implicit namespace package
sys.path = [p for p in sys.path if 'inftools' not in p]
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
from pprint import pprint    # to print the vars of the pathensemble object
import numpy as np
import glob

# Reading
from tistools import read_inputfile, get_LMR_interfaces, read_pathensemble, get_weights
from tistools import set_tau_distrib, set_tau_first_hit_M_distrib, cross_dist_distr, pathlength_distr
from tistools import collect_tau, collect_tau1, collect_tau2, collect_taum
from tistools import ACCFLAGS, REJFLAGS

# REPPTIS analysis
from tistools import get_lmr_masks, get_generation_mask, get_flag_mask, select_with_masks
from tistools import unwrap_by_weight, running_avg_local_probs, get_local_probs, get_global_probs_from_dict, get_global_probs_from_local

# MSM functions
from tistools import construct_M
from tistools import global_pcross_msm
from tistools import mfpt_to_first_last_state, mfpt_to_absorbing_states, construct_tau_vector
from tistools import create_labels_states, print_vector, print_all_tau

## 2. Load the simulation data

In [ ]:
# Set the working directory
# indir = "/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/APPTIS/simdata_kcl/infrepptis/"  
# indir = "/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis/" 
# indir = "/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/Z_pot2D/sim_retisZ"
# indir = "/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/Z_pot2D/sim_istarz"
# indir = "/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/maze2D/sim_istarmaze_kick2708"
# indir = "/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/maze2D/sim_retismaze0209"
# indir = "/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/other2D/sim_istarcffs4.2"
# indir = "/run/user/1001/gvfs/smb-share:server=files.ugent.be,share=eliawils,user=eliawils/shares/tw06_biommeda_pastime1/11.2024_StapleTIS_Elias/simulations/other2D/sim_istarcffs4.1"


# Control verbosity and options
VERBOSE = True          # Set True for detailed per-ensemble printing
PRINT_PATHENSEMBLE = True  # Set True to pprint each pathensemble vars

# zero_minus_one: True if lambda_-1 interface is set
zero_minus_one = False

# inputfile = indir + "/retis.rst"    # When using PyRETIS, the input file for REPPTIS simulations is a .rst file
inputfile = indir + "/repptis.rst"
# inputfile = indir + "/logging.log"

# Move to working directory
os.chdir(indir)
print(os.getcwd())

# Set the ensemble folders and print them
folders = sorted(glob.glob(indir + "/0[0-9][0-9]"))
print(f"Found {len(folders)} ensemble folders")
print(folders)


/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis
Found 16 ensemble folders
['/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis/000', '/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis/001', '/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis/002', '/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis/003', '/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis/004', '/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis/005', '/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis/006', '/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis/007', '/run/user/1001/gvfs/sftp:

In [ ]:
# Reading all input
#===================
interfaces, zero_left, timestep = read_inputfile(inputfile)
LMR_interfaces, LMR_strings = get_LMR_interfaces(interfaces, zero_left)

pathensembles = []
for i, fol in enumerate(folders):
    pe = read_pathensemble(fol + "/pathensemble.txt")
    pe.set_name(fol)
    pe.set_interfaces([LMR_interfaces[i], LMR_strings[i]])
    if i == 0:
        pe.set_zero_minus_one(zero_minus_one)   # TODO this is never used
        pe.set_in_zero_minus(True)
    if i == 1:
        pe.set_in_zero_plus(True)

    w, _ = get_weights(pe.flags, ACCFLAGS, REJFLAGS, verbose=False)
    pe.set_weights(w)

    if PRINT_PATHENSEMBLE:
        print("#" * 80)
        print(fol)
        print("pathensemble info:")
        pprint(vars(pe))
    else:
        print(f"Loaded {fol} (ensemble {i})")

    # Read order parameters order.txt/order.npy into path ensemble object, or load from order.npy file.
    # Saving order parameter files allows to speed up this notebook.
    pe.set_orders(load=False, acc_only=True, save=True)        # first run: store .npy files
    # pe.set_orders(load=True, acc_only=True, save=False)          # subsequent runs: read .npy files
    # pe.set_orders(load=False, acc_only=True, save=False)       # if saving doesn't work

    pathensembles.append(pe)

print(f"\nLoaded {len(pathensembles)} pathensembles")


['5', '1', '1', 'R', 'M', 'R', '1', 'REJ', 'sh', '-3.2209330000e+01', '-2.3946780000e+01', '0', '0', '0.0000000000e+00', '0', '0', '1.0000000000e+00']
[1. 1. 1. ... 1. 1. 1.]
################################################################################
/run/user/1001/gvfs/sftp:host=172.18.15.42,user=elias/home/elias/data/CG_P2C6_StapleTIS/infrepptis/000
pathensemble info:
{'cyclenumbers': array([    0,     1,     2, ..., 88565, 88566, 88567]),
 'dirs': array([0., 0., 0., ..., 0., 0., 0.]),
 'flags': array(['ACC', 'REJ', 'ACC', ..., 'REJ', 'REJ', 'REJ'], dtype='<U3'),
 'generation': array(['sh', 'sh', 'sh', ..., 'sh', 'sh', 'sh'], dtype='<U2'),
 'has_zero_minus_one': False,
 'in_zero_minus': True,
 'in_zero_plus': False,
 'interfaces': [[-30.0, -27.0, -24.0],
                ['l_[-1]', '( l_[-1] + l_[0] ) / 2', 'l_[0]']],
 'istar_idx': array([[0, 0],
       [0, 0],
       [0, 0],
       ...,
       [0, 0],
       [0, 0],
       [0, 0]]),
 'lambmaxs': array([-23.85022, -23.85022, -23.

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (29,) + inhomogeneous part.

: 

The `mlcolvar` object uses `DictDataset` extensively. Note that phasepoints aren't sequentially uniform because different paths jump around in time. Thus we pair configurations specifically $x_t \rightarrow x_{t+\tau}$ within the same path bounds:

In [4]:
print(np.mean(pathensembles[4].lengths))

475.96642033579667


In [5]:
import sys
import os

# Ensure tistools lib is in the path
lib_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if lib_path not in sys.path:
    sys.path.append(lib_path)

from tistools import prepare_tica_dataset, fit_mlcolvar_tica

# Choose lag time
lag_time = 300

# Prepare DictDataset directly from pathensembles using your updated logic
# `return_info=True` gives us extensive diagnostics on exactly what paths are used.
dataset, info = prepare_tica_dataset(
    pathensembles=pathensembles, 
    lag_time=lag_time,
    stride=10, 
    return_info=True
)

print(f"--- PATH USAGE DIAGNOSTICS ---")
print(f"Paths used:     {info['used_paths']} / {info['total_paths']} ({info['pct_paths_used']:.1f}%)")
print(f"Total Weight:   {info['used_weight']:.4g} / {info['total_weight']:.4g} ({info['pct_weight_used']:.1f}% weighted utilization)")
print(f"Frames mapped:  {len(dataset)} chunks inside DictDataset\n")

# Fit Mlcolvar TICA (configured to extract 2 output features for 2D plotting)
ew, ev, model = fit_mlcolvar_tica(dataset, n_cvs=2)

print(f"--- TICA COMPONENT QUALITY ---")
ew_np = ew.detach().cpu().numpy()
print(f"Eigenvalues (\u03BB_i): {ew_np}")

# Compute Implicit timescales : \u03C4_i  = - (lag_time) / ln(\u03BB_i)
ew_clipped = np.clip(ew_np, 1e-12, 1 - 1e-12) # safeguard log Domain
timescales = -lag_time / np.log(ew_clipped)

print("\nQuality Summary (Kinetic Variance & Timescale):")
for i, (val, ts) in enumerate(zip(ew_np, timescales)):
    print(f"  TIC {i+1} : Autocorrelation = {val:.4f} | Timescale = {ts:.1f} frames")
    print(f"    Eigenvector: {ev[:, i]}")

--- PATH USAGE DIAGNOSTICS ---
Paths used:     451262 / 1100011 (41.0%)
Total Weight:   8.292e+05 / 1.1e+06 (75.4% weighted utilization)
Frames mapped:  20670181 chunks inside DictDataset

--- TICA COMPONENT QUALITY ---
Eigenvalues (λ_i): [0.797158   0.37422788]

Quality Summary (Kinetic Variance & Timescale):
  TIC 1 : Autocorrelation = 0.7972 | Timescale = 1323.3 frames
    Eigenvector: tensor([0.7994, 0.6008])
  TIC 2 : Autocorrelation = 0.3742 | Timescale = 305.2 frames
    Eigenvector: tensor([ 0.2254, -0.9743])


In [10]:
# =========================================================================
# Ideal Lag Time Detection via Implied Timescales (\u03C4_i) Analysis Plot
# =========================================================================

lag_times_to_test = [20, 50, 100, 200, 300, 500, 700, 1000]
implied_timescales = []
valid_lags = []

for lt in lag_times_to_test:
    try:
        # Load dataset across iterations
        ds, _ = prepare_tica_dataset(pathensembles=pathensembles, lag_time=lt, pcross=None, stride=10, return_info=True)
        ew, _, _ = fit_mlcolvar_tica(ds, n_cvs=2)
        
        ew_np = ew.detach().cpu().numpy()
        ew_clipped = np.clip(ew_np, 1e-12, 1 - 1e-12)
        
        ts = -lt / np.log(ew_clipped)
        implied_timescales.append(ts)
        valid_lags.append(lt)
    except ValueError as e:
        print(f"Skipping lag_time={lt}: {e}")

implied_timescales = np.array(implied_timescales)

plt.figure(figsize=(9, 5))
for i in range(implied_timescales.shape[1]):
    plt.plot(valid_lags, implied_timescales[:, i], marker='o', lw=2, label=f'TIC {i+1}')

# Resolution limit: we can't reliably detect physical processes faster than the lag time itself 
plt.fill_between([0, max(valid_lags)], [0, max(valid_lags)], color='gray', alpha=0.2, label='Resolution limit (\u03C4 = lag)')

plt.xlabel('Lag Time \u03C4')
plt.ylabel('Implied Timescale \u03C4_i')
plt.xscale('log')
plt.yscale('log')
plt.title('Implied Timescales vs Lag Time to detect Markovianity')
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()
plt.show()

print("--> Interpretation:")
print("Look for the region where the plotted implied timescales LEVEL OFF (plateau horizontally).")
print("The ideal operational lag_time sits at the beginning of that plateau, indicating")
print("the process is now properly Markovian and decoupled from immediate memory.")

--> Interpretation:
Look for the region where the plotted implied timescales LEVEL OFF (plateau horizontally).
The ideal operational lag_time sits at the beginning of that plateau, indicating
the process is now properly Markovian and decoupled from immediate memory.


In [7]:
# Visualize the original order parameter for the first trajectory and its TICA projection
op0 = pathensembles[6].orders[5424]
if isinstance(op0, list):
    op0 = np.asarray(op0)
op_features = np.asarray(op0)

if op_features.ndim == 1:
    original_signal = op_features
    op_features = op_features.reshape(-1, 1)
else:
    original_signal = op_features[:, 0]

# Extract numpy array from torch.Tensor to allow matrix multiplication
if hasattr(ev, "detach"):
    ev_np = ev.detach().cpu().numpy()
else:
    ev_np = np.asarray(ev)

tica_proj = (op_features - op_features.mean(axis=0)) @ ev_np[:, 0]
tica_proj = np.asarray(tica_proj).ravel()

plt.figure(figsize=(11, 4))
plt.plot(original_signal, label='Original OP[0]', alpha=0.8)
plt.plot(tica_proj, label='TICA projection (first TIC)', alpha=0.8)
plt.xlabel('Frame index')
plt.ylabel('Value')
plt.title('Original order parameter vs TICA first component for path 0')
plt.legend()
plt.tight_layout()
plt.show()


In [19]:
# Plotting the Original and TICA transformed potential surfaces
import sys
import os
sys.path.append(os.path.join(indir, 'potentials'))

try:
    from sjoelbak import RectangularGridWithBarrierPotential
except Exception:
    RectangularGridWithBarrierPotential = None

try:
    from mazepotential_mixed import Maze2D_color
except Exception:
    Maze2D_color = None
try:
    from cffs2d import PotentialcFFS
except Exception:
    PotentialcFFS = None

# Choose the potential type here:
potential_kind = "sjoelbak"  # options: "barrier", "mazepotential", "cffs"


def get_potential_instance(kind):
    kind = kind.lower()
    if kind in ("barrier", "sjoelbak", "rectangular", "rectangulargridwithbarrierpotential"):
        if RectangularGridWithBarrierPotential is None:
            raise ImportError("RectangularGridWithBarrierPotential could not be imported.")
        return RectangularGridWithBarrierPotential(), "RectangularGridWithBarrierPotential"
    if kind in ("maze", "mazepotential", "maze2d", "maze2d_color"):
        if Maze2D_color is None:
            raise ImportError("Maze2D_color could not be imported.")
        return Maze2D_color(mazefig="potentials/maze.png"), "Maze2D_color"
    if kind in ("cffs",):
        if PotentialcFFS is None:
            raise ImportError("PotentialcFFS could not be imported.")
        return PotentialcFFS(4, np.pi/6), "PotentialcFFS"
    raise ValueError(f"Unknown potential kind: {kind}")


def get_potential_grid(pot, N=100, bounds=(0.0, 1.0, 0.0, 1.0)):
    if hasattr(pot, "get_potential_plot"):
        return pot.get_potential_plot(N=N)

    x_min, x_max, y_min, y_max = bounds
    xs = np.linspace(x_min, x_max, N)
    ys = np.linspace(y_min, y_max, N)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X, dtype=float)
    for i in range(N):
        for j in range(N):
            Z[i, j] = pot.potential(np.array([[Y[i, j], X[i, j]]]))
    return X, X, Z


pot, pot_name = get_potential_instance(potential_kind)
print(f"Using potential: {pot_name}")

# xvals, yvals, potvals = get_potential_grid(pot, N=100, bounds=(0.1, 0.93, 0.15, 0.78))
yvals, xvals, potvals = get_potential_grid(pot, N=100, bounds=(-6.5, 6.5, -6.5, 6.5))

# 1. Create a grid in the original (x, y) space
# 2. Transform the grid and path to TICA space
grid_pts = np.c_[xvals.ravel(), yvals.ravel()]
grid_pts_centered = grid_pts - op_features.mean(axis=0)

tic1_grid = grid_pts_centered @ ev_np[:, 0]

op_features_centered = op_features - op_features.mean(axis=0)
if op_features.ndim == 1:
    op_features = op_features.reshape(-1, 1)

tic1_path = op_features_centered @ ev_np[:, 0]

if ev_np.shape[1] > 1:
    tic2_grid = grid_pts_centered @ ev_np[:, 1]
    tic2_path = op_features_centered @ ev_np[:, 1]
    y_label_2 = 'TIC 2'
else:
    tic2_grid = yvals.ravel()
    tic2_path = op_features_centered[:, 0]
    y_label_2 = 'Original Y (TIC 2 missing)'

mask = np.isfinite(potvals.ravel())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cntr1 = axes[0].tricontourf(xvals.ravel()[mask], yvals.ravel()[mask], potvals.ravel()[mask], levels=20, cmap='viridis')
fig.colorbar(cntr1, ax=axes[0], label='Potential U(x, y)')
if op_features.shape[1] > 1:
    axes[0].plot(op_features[:, 0], op_features[:, 1], color='red', alpha=0.8, linewidth=2, label='Path 0')

# Add TICA direction arrows in original coordinate space
if op_features.shape[1] > 1:
    origin = op_features.mean(axis=0)
    scale = max(np.ptp(op_features[:, 0]), np.ptp(op_features[:, 1]), 1.0) * 0.4
    arrows = np.column_stack((ev_np[:, 0]/abs(ev_np[:, 0]), ev_np[:, 1]/abs(ev_np[:, 1]))) * scale
    axes[0].quiver(
        [origin[0], origin[0]],
        [origin[1], origin[1]],
        arrows[0, :],
        arrows[1, :],
        angles='xy',
        scale_units='xy',
        scale=1/scale,
        color=["white", "yellow"],
        width=0.005,
        label='TIC directions'
    )
    axes[0].annotate('TIC1', xy=(origin[0] + arrows[0, 0] * scale, origin[1] + arrows[1, 0] * scale), xytext=(5, 5), textcoords='offset points', color='white')
    if ev_np.shape[1] > 1:
        axes[0].annotate('TIC2', xy=(origin[0] + arrows[0, 1] * scale, origin[1] + arrows[1, 1] * scale), xytext=(5, -12), textcoords='offset points', color='yellow')

axes[0].set_xlabel('Original X')
axes[0].set_ylabel('Original Y')
axes[0].set_title(f'Original Coordinates ({pot_name})')
axes[0].legend()

cntr2 = axes[1].tricontourf(tic1_grid[mask], tic2_grid[mask], potvals.ravel()[mask], levels=20, cmap='viridis')
fig.colorbar(cntr2, ax=axes[1], label='Potential U in TICA space')
axes[1].plot(tic1_path, tic2_path, color='red', alpha=0.8, linewidth=2, label='Path 0')
axes[1].set_xlabel('TIC 1')
axes[1].set_ylabel(y_label_2)
axes[1].set_title('TICA Coordinates')
axes[1].legend()

plt.tight_layout()
plt.show()

Using potential: RectangularGridWithBarrierPotential


### Transitioning to DeepTICA

To transition this logic directly to `DeepTICA` in `mlcolvar`, you just pass your dataset to their lightning wrappers. This prepares weighted subsets identically and safely for Deep Learning:

```python
from mlcolvar.cvs import DeepTICA
from mlcolvar.data import DictModule
import lightning as pl

datamodule = DictModule(dataset, lengths=[0.8, 0.2]) # validation splits
deep_tica = DeepTICA(layers=[2, 10, 10, 1])

trainer = pl.Trainer(max_epochs=100)
trainer.fit(deep_tica, datamodule)
```

### Multi-Task DeepTICA with Turn-Depth Regularization

We can use the `MultiTaskDeepTICA` CV along with a `DictModule` to load the two tasks.
First, we prepare the two datasets.

In [20]:
del dataset

In [ ]:
import torch
from torch.utils.data import Subset
from mlcolvar.data import DictLoader
import math
from tistools.tica import MultiTaskDeepTICA, prepare_multitask_datasets

dataset_tica, dataset_depth = prepare_multitask_datasets(pathensembles=pathensembles, lag_time=lag_time, stride=10)

L_tica = len(dataset_tica)
L_depth = len(dataset_depth)

# 1) Split 80/20 train/val
L_tica_train = int(L_tica * 0.8)
L_tica_val = L_tica - L_tica_train

L_depth_train = int(L_depth * 0.8)
L_depth_val = L_depth - L_depth_train

# 2) Fixed TICA Batch Sizes -> gives us precise total Target Batches
b_tica_train = 5000000
b_tica_val = 5000000

# TICA Batches
n_train_batches = math.ceil(L_tica_train / b_tica_train)
n_val_batches = math.ceil(L_tica_val / b_tica_val)

# 3) Compute necessary Depth Batch Sizes and Total padded sizes to match Target Batches exactly
b_depth_train = math.ceil(L_depth_train / n_train_batches)
target_train_padded_size = n_train_batches * b_depth_train

b_depth_val = math.ceil(L_depth_val / n_val_batches)
target_val_padded_size = n_val_batches * b_depth_val

print("--- Data Layout ---")
print(f"TICA:  Total={L_tica}, Train={L_tica_train}, Val={L_tica_val}")
print(f"DEPTH: Total={L_depth}, Train={L_depth_train}, Val={L_depth_val}")
print("\n--- Math ---")
print(f"Target Train Batches: {n_train_batches}")
print(f"Target Val Batches:   {n_val_batches}")

# 4) Pad Depth indices to make them exactly integer-divisible into the same n_batches
pad_train_amount = target_train_padded_size - L_depth_train
pad_val_amount = target_val_padded_size - L_depth_val

print(f"\nPadding Depth Train by {pad_train_amount} items (to size {target_train_padded_size})")
print(f"Padding Depth Val by   {pad_val_amount} items (to size {target_val_padded_size})")

train_depth_indices = list(range(0, L_depth_train)) + list(range(0, pad_train_amount))
val_depth_indices = list(range(L_depth_train, L_depth)) + list(range(L_depth_train, L_depth_train + pad_val_amount))

# 5) Create exact Subsets
ds_tica_train = Subset(dataset_tica, range(0, L_tica_train))
ds_tica_val = Subset(dataset_tica, range(L_tica_train, L_tica))

ds_depth_train = Subset(dataset_depth, train_depth_indices)
ds_depth_val = Subset(dataset_depth, val_depth_indices)

# 6) Create DictLoaders directly
t_loader = DictLoader([ds_tica_train, ds_depth_train], batch_size=[b_tica_train, b_depth_train], shuffle=True)
v_loader = DictLoader([ds_tica_val, ds_depth_val], batch_size=[b_tica_val, b_depth_val], shuffle=True)

print("\nValidating DictLoader batch sizes...")
print(f"Train Dataloader batches = {len(t_loader)}")
print(f"Valid Dataloader batches = {len(v_loader)}")

assert len(t_loader) == n_train_batches, "Train batch count mismatch!"
assert len(v_loader) == n_val_batches, "Val batch count mismatch!"
print("✅ Success! Datasets are perfectly synchronized for mlcolvar's DictLoader.")

--- Data Layout ---
TICA:  Total=12703759, Train=10163007, Val=2540752
DEPTH: Total=334642, Train=267713, Val=66929

--- Math ---
Target Train Batches: 3
Target Val Batches:   1

Padding Depth Train by 1 items (to size 267714)
Padding Depth Val by   0 items (to size 66929)

Validating DictLoader batch sizes...
Train Dataloader batches = 3
Valid Dataloader batches = 1
✅ Success! Datasets are perfectly synchronized for mlcolvar's DictLoader.


In [33]:
from tistools.tica import MultiTaskDeepTICA
import lightning as pl
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from mlcolvar.utils.trainer import MetricsCallback
from lightning.pytorch.loggers import CSVLogger
from mlcolvar.utils.plot import plot_isolines_2D, plot_metrics
import os

# define callbacks
metrics = MetricsCallback()
# We have 2 features down to 2 TICs. We map our nn depth with similar layers.
options = {'nn': {'activation': 'shifted_softplus'}}
deep_tica = MultiTaskDeepTICA(layers=[2, 15, 15, 5], n_cvs=2, alpha=0, warmup_steps=0, loss_type="mse", options=options)
# Increase Cholesky regularization to make the covariance matrix positive-definite
# deep_tica.tica.reg_C_0 = 1e-3

early_stop_callback = EarlyStopping(
    monitor="valid_loss", 
    min_delta=1e-5, 
    patience=20, 
    verbose=True, 
    mode="min"
)

# Force the logger to write to a safe local directory!
log_dir = "/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/APPTIS/tistools/lightning_logs"
os.makedirs(log_dir, exist_ok=True)
local_logger = CSVLogger(save_dir=log_dir, name="multitask_tica")

trainer = pl.Trainer(
    max_epochs=None,
    callbacks=[metrics, early_stop_callback],
    accelerator="gpu",
    devices=1,
    log_every_n_steps=10,
    logger=local_logger,
    enable_model_summary=False
)

# Initialize normalization properly before fit bypassing setup key errors
# using our custom patched setup block
try:
    deep_tica.setup(stage="fit")
except Exception as e:
    print("Normal lightning datamodule skipped:", e)
    pass

# Force data normalization stats from our dataset manually
stats = ds_tica_train.dataset.get_stats()['data']
deep_tica.norm_in.set_from_stats(stats)
deep_tica.norm_in.is_initialized = True

print("Normalization In Stats Assigned manually.")

# FIT THE MODEL
trainer.fit(deep_tica, t_loader, v_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Normal lightning datamodule skipped: MultiTaskDeepTICA is not attached to a `Trainer`.
KEY:  data


KEY:  data_lag


KEY:  weights


KEY:  weights_lag




/home/elias/anaconda3/envs/test/lib/python3.12/site-packages/lightning/pytorch/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

/home/elias/anaconda3/envs/test/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:79: Trying to 
infer the `batch_size` from an ambiguous collection. The batch size we found is 2540752. To avoid any 
miscalculations, use `self.log(..., batch_size=batch_size)`.

Normalization In Stats Assigned manually.


/home/elias/anaconda3/envs/test/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:321: The number of
training batches (3) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for 
log_every_n_steps if you want to see logs for the training epoch.

/home/elias/anaconda3/envs/test/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:79: Trying to 
infer the `batch_size` from an ambiguous collection. The batch size we found is 163007. To avoid any 
miscalculations, use `self.log(..., batch_size=batch_size)`.

Metric valid_loss improved. New best score: -0.086


Monitored metric valid_loss did not improve in the last 20 records. Best score: -0.086. Signaling Trainer to stop.


In [34]:
ax = plot_metrics(metrics.metrics,
                  keys=[x for x in  metrics.metrics.keys() if 'valid_eigval' in x],#['train_loss_epoch','valid_loss'],
                #   linestyles=['-.','-'], colors=['fessa1','fessa5'],
                  yscale='linear')

In [38]:
from mlcolvar.core.transform import Normalization
from mlcolvar.core.transform.utils import Statistics

n_components = 2
X = dataset_tica[:]['data']
with torch.no_grad():
    model.postprocessing = None # reset
    s = model(torch.Tensor(X))

norm =  Normalization(n_components, mode='min_max', stats = Statistics(s) )
model.postprocessing = norm

fig,axs = plt.subplots( 1, n_components, figsize=(5*n_components,4) )

if n_components == 1:
    axs = [axs]
for i in range(n_components):
    ax = axs[i]
    ax.set_ylim(-.02,0.37)
    ax.set_xlim(-0.1,1)

    # plot_isolines_2D(pot,levels=np.linspace(0,24,12),mode='contour',ax=ax)
    ax.contour(xvals, yvals, potvals, levels=12, colors='black', alpha=0.3, linewidths=1.2)
    plot_isolines_2D(deep_tica, component=i, levels=np.linspace(-0.05,0.1,30), ax=ax, limits=((-6, 6), (-6, 6)))
    #ax.scatter(X[:,0],X[:,1],s=1, alpha=0.2,c='w')


In [41]:
# Evaluate DeepTICA on path 0

import torch

deep_tica.eval()
op0 = pathensembles[4].orders[85454]
if isinstance(op0, list):
    op0 = np.asarray(op0)
op_features = np.asarray(op0)

if op_features.ndim == 1:
    original_signal = op_features
    op_features = op_features.reshape(-1, 1)
else:
    original_signal = op_features[:, 0]

# Pass through DeepTICA (moves it to CPU and applies detach)
with torch.no_grad():
    in_tensor = torch.tensor(op_features, dtype=torch.float32, device=deep_tica.device)
    deeptica_proj = deep_tica(in_tensor).cpu().numpy()

# Note: DeepTICA output shape is (N_frames, n_cvs)
deeptica_tic1 = deeptica_proj[:, 0]

plt.figure(figsize=(11, 4))
plt.plot(original_signal, label='Original OP[0]', alpha=0.8)
plt.plot(deeptica_tic1, label='DeepTICA projection (first TIC)', alpha=0.8)
plt.xlabel('Frame index')
plt.ylabel('Value')
plt.title('Original order parameter vs DeepTICA first component for path 0')
plt.legend()
plt.tight_layout()
plt.show()

In [42]:
# Plot Deep TICA Potential Surface

# Option to plot TIC 1 vs TIC 2 (True) or TIC 1 vs Original Y (False)
plot_tic2 = True

# 1. Transform the grid and path through DeepTICA
with torch.no_grad():
    grid_tensor = torch.tensor(grid_pts, dtype=torch.float32, device=deep_tica.device)
    deeptica_grid = deep_tica(grid_tensor).cpu().numpy()
    
tic1_grid_deep = deeptica_grid[:, 0]
tic1_path_deep = deeptica_proj[:, 0]

# Configure the Y-axis of the transformed plot based on the toggle flag
if plot_tic2 and deeptica_grid.shape[1] > 1:
    tic2_grid_deep = deeptica_grid[:, 1]
    tic2_path_deep = deeptica_proj[:, 1]
    y_label_2_deep = 'DeepTICA TIC 2'
else:
    tic2_grid_deep = yvals.ravel()
    tic2_path_deep = op_features[:, 1] if op_features.shape[1] > 1 else np.zeros_like(tic1_path_deep)
    y_label_2_deep = 'Original Y'

mask = np.isfinite(potvals.ravel())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cntr1 = axes[0].tricontourf(xvals.ravel()[mask], yvals.ravel()[mask], potvals.ravel()[mask], levels=20, cmap='viridis')
fig.colorbar(cntr1, ax=axes[0], label='Potential U(x, y)')
if op_features.shape[1] > 1:
    axes[0].plot(op_features[:, 0], op_features[:, 1], color='red', alpha=0.8, linewidth=2, label='Path 0')

# =========================================================================
# To visualize the non-linear DeepTICA mapping, plot unit gradient vectors 
# (directions) and coordinate mesh contours of the TICs
# =========================================================================
tic1_reshaped = tic1_grid_deep.reshape(xvals.shape)
# Always compute actual TIC2 gradient for overlay curves in the left plot
tic2_actual_reshaped = deeptica_grid[:, 1].reshape(xvals.shape) if deeptica_grid.shape[1] > 1 else np.zeros_like(tic1_reshaped)

cs1 = axes[0].contour(xvals, yvals, tic1_reshaped, levels=12, colors='white', alpha=0.3, linewidths=1.2)
dy1, dx1 = np.gradient(tic1_reshaped) # Note np.gradient gives (axis 0, axis 1) -> (y, x) directions for meshgrid
norm1 = np.hypot(dx1, dy1)
norm1[norm1 == 0] = 1
axes[0].quiver(xvals[::5, ::5], yvals[::5, ::5], 
               (dx1/norm1)[::5, ::5], (dy1/norm1)[::5, ::5], 
               color='white', alpha=0.8, scale=30, headwidth=5)

if deeptica_grid.shape[1] > 1:
    cs2 = axes[0].contour(xvals, yvals, tic2_actual_reshaped, levels=12, colors='yellow', alpha=0.3, linewidths=1.2)
    dy2, dx2 = np.gradient(tic2_actual_reshaped)
    norm2 = np.hypot(dx2, dy2)
    norm2[norm2 == 0] = 1
    # axes[0].quiver(xvals[::5, ::5], yvals[::5, ::5], 
    #                (dx2/norm2)[::5, ::5], (dy2/norm2)[::5, ::5], 
    #                color='yellow', alpha=0.8, scale=30, headwidth=5)

axes[0].plot([], [], color='white', label='TIC 1 Direction & Curves')
axes[0].plot([], [], color='yellow', label='TIC 2 Direction & Curves')

axes[0].set_xlabel('Original X')
axes[0].set_ylabel('Original Y')
axes[0].set_title(f'Original Coordinates ({pot_name})')
axes[0].legend()

cntr2 = axes[1].tricontourf(tic1_grid_deep[mask], tic2_grid_deep[mask], potvals.ravel()[mask], levels=20, cmap='viridis')
fig.colorbar(cntr2, ax=axes[1], label='Potential U in DeepTICA space' if plot_tic2 else 'Potential U (TIC1, Orig-Y)')
axes[1].plot(tic1_path_deep, tic2_path_deep, color='red', alpha=0.8, linewidth=2, label='Path 0')
axes[1].set_xlabel('DeepTICA TIC 1')
axes[1].set_ylabel(y_label_2_deep)
axes[1].set_title('DeepTICA Coordinates' if plot_tic2 else 'TIC 1 Projection vs Original Y')
axes[1].legend()

plt.tight_layout()
plt.show()

In [43]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# The CSVLogger saves to a specific version folder (trainer.logger.log_dir)
metrics_file = os.path.join(local_logger.log_dir, "metrics.csv")
metrics = pd.read_csv(metrics_file)

print("Metrics tracked by the logger:")
print(metrics.columns.tolist())
    
# Create a clean plot for all loss components
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_cols = [c for c in metrics.columns if 'train' in c and 'loss' in c]
valid_cols = [c for c in metrics.columns if 'valid' in c and 'loss' in c]

# Group by epoch to average out any batch/step fluctuations
for c in train_cols:
    df_clean = metrics[['epoch', c]].dropna()
    df_clean = df_clean.groupby('epoch').mean().reset_index()
    axes[0].plot(df_clean['epoch'], df_clean[c], label=c, lw=2)

axes[0].set_title('Training Losses')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

for c in valid_cols:
    df_clean = metrics[['epoch', c]].dropna()
    df_clean = df_clean.groupby('epoch').mean().reset_index()
    axes[1].plot(df_clean['epoch'], df_clean[c], label=c, lw=2)

axes[1].set_title('Validation Losses')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

Metrics tracked by the logger:
['epoch', 'step', 'train_eigval_1_epoch', 'train_eigval_1_step', 'train_eigval_2_epoch', 'train_eigval_2_step', 'train_loss_depth_epoch', 'train_loss_depth_step', 'train_loss_epoch', 'train_loss_step', 'train_loss_tica_epoch', 'train_loss_tica_step', 'train_tica_weight_epoch', 'train_tica_weight_step', 'valid_eigval_1_epoch', 'valid_eigval_1_step', 'valid_eigval_2_epoch', 'valid_eigval_2_step', 'valid_loss_depth_epoch', 'valid_loss_depth_step', 'valid_loss_epoch', 'valid_loss_step', 'valid_loss_tica_epoch', 'valid_loss_tica_step', 'valid_tica_weight_epoch', 'valid_tica_weight_step']
